In [0]:
from pyspark.sql.functions import col, trim, upper, row_number
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# ==========================================
# 04 - GENERIC SILVER TRANSFORMATION
# ==========================================

configs = (
    spark.table("mortgage_catalog.metadata.silver_config")
    .filter("is_active = true")
    .collect()
)

for config in configs:

    source_table = config["source_table"]
    target_table = config["target_table"]
    primary_key  = config["primary_key"]
    order_column = config["order_column"]

    source = f"mortgage_catalog.silver.{source_table}"
    target = f"mortgage_catalog.silver.{target_table}"

    print(f"\nProcessing → {source}")

    # ----------------------------------
    # Check source exists
    # ----------------------------------
    if not spark.catalog.tableExists(source):
        print(f"SKIPPED → {source} not found")
        continue

    df = spark.table(source)

    # ----------------------------------
    # Clean STRING columns
    # Trim + Uppercase
    # ----------------------------------
    for field in df.schema.fields:

        if (
            isinstance(field.dataType, StringType)
            and not field.name.startswith("_")
            and field.name not in ["dq_error"]
        ):

            df = df.withColumn(
                field.name,
                upper(trim(col(field.name)))
            )

    # ----------------------------------
    # Remove duplicates
    # Keep latest record
    # ----------------------------------
    window_spec = (
        Window
        .partitionBy(primary_key)
        .orderBy(col(order_column).desc())
    )

    df = (
        df
        .withColumn("_row_number", row_number().over(window_spec))
        .filter(col("_row_number") == 1)
        .drop("_row_number")
    )

    # ----------------------------------
    # Write clean Silver
    # ----------------------------------
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    print(f"SUCCESS → {target}")

print("\nALL SILVER TRANSFORMATIONS COMPLETED ✅")


Processing → mortgage_catalog.silver.property_silver
SUCCESS → mortgage_catalog.silver.property_clean

Processing → mortgage_catalog.silver.branch_silver
SUCCESS → mortgage_catalog.silver.branch_clean

Processing → mortgage_catalog.silver.document_silver
SUCCESS → mortgage_catalog.silver.document_clean

Processing → mortgage_catalog.silver.loan_application_silver
SUCCESS → mortgage_catalog.silver.loan_application_clean

Processing → mortgage_catalog.silver.payment_silver
SUCCESS → mortgage_catalog.silver.payment_clean

Processing → mortgage_catalog.silver.credit_score_silver
SUCCESS → mortgage_catalog.silver.credit_score_clean

Processing → mortgage_catalog.silver.customer_autoloader
SUCCESS → mortgage_catalog.silver.customer_clean

ALL SILVER TRANSFORMATIONS COMPLETED ✅
